# amusnaw-mt : entraîner notre traducteur français ↔ kabyle

Ce carnet part de **NLLB-200 (600M)** de Meta et le réentraîne sur nos données :
1. **Rétro-traduction** : du vrai kabyle humain est traduit vers le français par NLLB → paires « français approximatif → kabyle authentique ».
2. **Entraînement** sur ces paires + nos ~240 000 paires humaines (Tatoeba, livres, contes, expressions, dictionnaire), dans les deux sens.
3. **Banc d'essai** : les mêmes 300 phrases que sur le serveur, avant et après. On garde le modèle seulement s'il fait mieux.
4. **Export** au format du serveur (CTranslate2 int8) : `amusnaw-mt-ct2.zip` dans l'onglet *Output*.

**Avant de lancer (panneau de droite) :**
- *Accelerator* : **GPU T4 x2** (ou P100)
- *Internet* : **On** (pour télécharger NLLB)
- *Add Input* : ton jeu de données **privé** `amusnaw-mt-data` (le zip fait par `build_mt_dataset.py`)

Puis *Save Version → Save & Run All (Commit)* : le carnet tourne seul jusqu'à 12 h, même navigateur fermé.

In [ ]:
!pip install -q transformers==4.38.2 ctranslate2==4.4.0 sentencepiece==0.2.0

In [ ]:
# ---------- Réglages ----------
BASE_MODEL = "facebook/nllb-200-distilled-600M"
N_BACKTRANSLATE = 300_000   # phrases kabyles rétro-traduites (le reste du corpus attendra la prochaine fois)
MAX_TRAIN_HOURS = 6.5       # l'entraînement s'arrête proprement après ce temps (séance Kaggle : 12 h max)
EPOCHS = 1
BATCH = 16
GRAD_ACCUM = 2
LR = 1e-4
WARMUP = 1000
MAX_LEN = 128
UPSAMPLE = {"livres": 3, "contes": 3, "expressions": 2}  # données rares mais précieuses : vues plusieurs fois
SEED = 2026

import glob, json, math, os, random, re, time, zipfile
import torch
from transformers import AutoModelForSeq2SeqLM, AutoTokenizer

random.seed(SEED); torch.manual_seed(SEED)
DEV = "cuda"
OUT = "/kaggle/working"
CODES = {"fr": "fra_Latn", "kab": "kab_Latn"}
T0 = time.time()

def find(name):
    hits = glob.glob(f"/kaggle/input/**/{name}", recursive=True)
    if not hits:  # zip non décompressé
        for z in glob.glob("/kaggle/input/**/*.zip", recursive=True):
            zipfile.ZipFile(z).extractall("/kaggle/working/data")
        hits = glob.glob(f"/kaggle/working/data/**/{name}", recursive=True)
    assert hits, f"{name} introuvable : ajoute le jeu de données amusnaw-mt-data"
    return hits[0]

def read_jsonl(p):
    with open(p, encoding="utf-8") as f:
        return [json.loads(l) for l in f if l.strip()]

train_pairs = read_jsonl(find("pairs_train.jsonl"))
dev_pairs = read_jsonl(find("pairs_dev.jsonl"))
eval_set = read_jsonl(find("eval_set.jsonl"))
mono = [l.strip() for l in open(find("mono_kab.txt"), encoding="utf-8") if l.strip()]
print(len(train_pairs), "paires humaines |", len(dev_pairs), "contrôle |", len(eval_set), "banc d'essai |", len(mono), "phrases kabyles")
print(torch.cuda.get_device_name(0))

In [ ]:
tok = AutoTokenizer.from_pretrained(BASE_MODEL)
model = AutoModelForSeq2SeqLM.from_pretrained(BASE_MODEL).to(DEV)

@torch.no_grad()
def translate(texts, src, tgt, beams=4, batch=48, max_new=160):
    """Traduit une liste de phrases (triées par longueur pour aller vite)."""
    model.eval()
    tok.src_lang = CODES[src]
    order = sorted(range(len(texts)), key=lambda i: len(texts[i]))
    out = [None] * len(texts)
    bos = tok.convert_tokens_to_ids(CODES[tgt])
    for i in range(0, len(order), batch):
        idx = order[i:i + batch]
        enc = tok([texts[j] for j in idx], return_tensors="pt", padding=True, truncation=True, max_length=MAX_LEN).to(DEV)
        with torch.autocast("cuda", dtype=torch.float16):
            gen = model.generate(**enc, forced_bos_token_id=bos, num_beams=beams, max_new_tokens=max_new,
                                 no_repeat_ngram_size=4)
        for j, t in zip(idx, tok.batch_decode(gen, skip_special_tokens=True)):
            out[j] = t.replace("ε", "ɛ").replace("γ", "ɣ")
    return out

from collections import Counter
def chrf(hyp, ref, n=6, beta=2.0):
    """Même calcul que le serveur (app/eval_translate.py) : les scores sont comparables."""
    h, r = hyp.replace(" ", ""), ref.replace(" ", "")
    P, R = [], []
    for k in range(1, n + 1):
        hc = Counter(h[i:i + k] for i in range(len(h) - k + 1)); rc = Counter(r[i:i + k] for i in range(len(r) - k + 1))
        if not hc or not rc:
            continue
        m = sum((hc & rc).values()); P.append(m / sum(hc.values())); R.append(m / sum(rc.values()))
    if not P:
        return 0.0
    p, rr = sum(P) / len(P), sum(R) / len(R)
    return 0.0 if p + rr == 0 else 100 * (1 + beta ** 2) * p * rr / (beta ** 2 * p + rr)

def bench(tag):
    """chrF sur le banc d'essai (mêmes phrases que sur le serveur)."""
    res = {}
    for src, tgt in (("fr", "kab"), ("kab", "fr")):
        hyp = translate([p[src] for p in eval_set], src, tgt)
        for subset in ("quotidien", "livres"):
            ids = [i for i, p in enumerate(eval_set) if p["set"] == subset]
            score = sum(chrf(hyp[i], eval_set[i][tgt]) for i in ids) / len(ids)
            res[f"{src}→{tgt} | {subset}"] = round(score, 1)
    print(tag, json.dumps(res, ensure_ascii=False, indent=1))
    return res

before = bench("NLLB de base")

## 1. Rétro-traduction

Le kabyle reste celui des humains ; seul le côté français est produit par la machine.

In [ ]:
bt_path = f"{OUT}/backtranslated.jsonl"
done = read_jsonl(bt_path) if os.path.exists(bt_path) else []
todo = mono[len(done):N_BACKTRANSLATE]
print("déjà faites :", len(done), "| à faire :", len(todo))
CHUNK = 20_000
with open(bt_path, "a", encoding="utf-8") as f:
    for i in range(0, len(todo), CHUNK):
        part = todo[i:i + CHUNK]
        frs = translate(part, "kab", "fr", beams=1, batch=128, max_new=128)  # glouton : 4× plus rapide
        for kab, fr in zip(part, frs):
            f.write(json.dumps({"fr": fr, "kab": kab}, ensure_ascii=False) + "\n")
        print(f"{len(done) + i + len(part)} / {N_BACKTRANSLATE}  ({(time.time() - T0) / 60:.0f} min)")

def ok(fr, kab):
    """Écarte les rétro-traductions ratées (vides, répétitives, de longueur absurde)."""
    if not fr or len(fr) < 3:
        return False
    words = fr.split()
    if len(words) >= 6 and len(set(words)) / len(words) < 0.5:
        return False
    return 0.4 <= (len(fr) + 5) / (len(kab) + 5) <= 2.5

bt = [r for r in read_jsonl(bt_path) if ok(r["fr"], r["kab"])]
print(len(bt), "paires rétro-traduites gardées")

## 2. Entraînement

In [ ]:
def encode(pairs, src, tgt, origin):
    tok.src_lang, tok.tgt_lang = CODES[src], CODES[tgt]
    out = []
    for i in range(0, len(pairs), 4096):
        part = pairs[i:i + 4096]
        enc = tok([p[src] for p in part], text_target=[p[tgt] for p in part], truncation=True, max_length=MAX_LEN)
        out += [(a, b, origin) for a, b in zip(enc["input_ids"], enc["labels"])]
    return out

examples = []
by_origin = {}
for p in train_pairs:
    by_origin.setdefault(p["origin"], []).append(p)
for origin, ps in by_origin.items():
    ps = ps * UPSAMPLE.get(origin, 1)
    examples += encode(ps, "fr", "kab", origin) + encode(ps, "kab", "fr", origin)
examples += encode(bt, "fr", "kab", "retro")  # rétro-traduction : seulement vers le kabyle
random.shuffle(examples)
print(len(examples), "exemples d'entraînement")

# lots de longueurs voisines (moins de remplissage = plus rapide)
def batches(exs):
    idx = list(range(len(exs)))
    random.shuffle(idx)
    chunks = [sorted(idx[i:i + BATCH * 100], key=lambda j: len(exs[j][0])) for i in range(0, len(idx), BATCH * 100)]
    bs = [c[i:i + BATCH] for c in chunks for i in range(0, len(c), BATCH)]
    random.shuffle(bs)
    return bs

def collate(ids):
    src = [examples[i][0] for i in ids]; lab = [examples[i][1] for i in ids]
    L, M = max(map(len, src)), max(map(len, lab))
    pad = tok.pad_token_id
    x = torch.tensor([s + [pad] * (L - len(s)) for s in src])
    y = torch.tensor([t + [-100] * (M - len(t)) for t in lab])
    return x.to(DEV), (x != pad).long().to(DEV), y.to(DEV)

dev_ex = encode(dev_pairs, "fr", "kab", "dev")[:500]
@torch.no_grad()
def dev_loss():
    model.eval(); tot = 0
    for i in range(0, len(dev_ex), 32):
        part = dev_ex[i:i + 32]
        L, M = max(len(a) for a, _, _ in part), max(len(b) for _, b, _ in part)
        x = torch.tensor([a + [tok.pad_token_id] * (L - len(a)) for a, _, _ in part]).to(DEV)
        y = torch.tensor([b + [-100] * (M - len(b)) for _, b, _ in part]).to(DEV)
        with torch.autocast("cuda", dtype=torch.float16):
            tot += model(input_ids=x, attention_mask=(x != tok.pad_token_id).long(), labels=y).loss.item() * len(part)
    model.train()
    return tot / len(dev_ex)

opt = torch.optim.AdamW(model.parameters(), lr=LR, weight_decay=0.01)
plan = batches(examples) * EPOCHS
total_steps = len(plan) // GRAD_ACCUM
sched = torch.optim.lr_scheduler.LambdaLR(opt, lambda s: min(1, (s + 1) / WARMUP) * max(0.05, 1 - s / total_steps))
scaler = torch.cuda.amp.GradScaler()
print("pas d'optimisation prévus :", total_steps, "| perte de contrôle au départ :", round(dev_loss(), 3))

model.train(); t_train = time.time(); step = 0; run = 0
for k, ids in enumerate(plan):
    x, m, y = collate(ids)
    with torch.autocast("cuda", dtype=torch.float16):
        loss = model(input_ids=x, attention_mask=m, labels=y).loss / GRAD_ACCUM
    scaler.scale(loss).backward()
    run += loss.item()
    if (k + 1) % GRAD_ACCUM == 0:
        scaler.unscale_(opt)
        torch.nn.utils.clip_grad_norm_(model.parameters(), 1.0)
        scaler.step(opt); scaler.update(); opt.zero_grad(set_to_none=True); sched.step(); step += 1
        if step % 500 == 0:
            h = (time.time() - t_train) / 3600
            print(f"pas {step}/{total_steps}  perte {run / 500:.3f}  contrôle {dev_loss():.3f}  {h:.2f} h", flush=True)
            run = 0
        if (time.time() - t_train) / 3600 > MAX_TRAIN_HOURS:
            print("Temps écoulé : on s'arrête ici et on garde ce qui est appris.")
            break

## 3. Banc d'essai et export

In [ ]:
after = bench("amusnaw-mt")
print("\nProgression (points chrF) :")
for k in before:
    print(f"  {k:24s} {before[k]:5.1f} → {after[k]:5.1f}  ({after[k] - before[k]:+.1f})")

tests = ["Ma grand-mère prépare le couscous le vendredi.",
         "Bonjour, je m'appelle Koceila et j'habite à Tizi Ouzou.",
         "La Cigale, ayant chanté tout l'été, se trouva fort dépourvue quand la bise fut venue."]
for s, t in zip(tests, translate(tests, "fr", "kab")):
    print(s, "\n  →", t)
json.dump({"avant": before, "apres": after, "pas": step}, open(f"{OUT}/rapport.json", "w"), ensure_ascii=False, indent=1)

In [ ]:
# Modèle au format du serveur (CTranslate2 int8)
import ctranslate2, shutil
model.save_pretrained(f"{OUT}/hf"); tok.save_pretrained(f"{OUT}/hf")
ctranslate2.converters.TransformersConverter(f"{OUT}/hf").convert(f"{OUT}/amusnaw-mt", quantization="int8", force=True)
tok.save_pretrained(f"{OUT}/amusnaw-mt")
shutil.copy(f"{OUT}/rapport.json", f"{OUT}/amusnaw-mt/rapport.json")
shutil.make_archive(f"{OUT}/amusnaw-mt-ct2", "zip", f"{OUT}/amusnaw-mt")
shutil.rmtree(f"{OUT}/hf"); shutil.rmtree(f"{OUT}/amusnaw-mt")
print("Prêt :", f"{OUT}/amusnaw-mt-ct2.zip", f"({os.path.getsize(f'{OUT}/amusnaw-mt-ct2.zip') / 1e6:.0f} Mo)",
      f"— durée totale {(time.time() - T0) / 3600:.1f} h")